# 📉 Customer Churn Prediction
### Telecom Dataset — End-to-End ML with Explainability

**Goal:** Predict which customers are likely to cancel their subscription, explain *why*, and translate findings into a business recommendation.

**Stack:** Python · scikit-learn · XGBoost · SHAP · Matplotlib · Seaborn

**Dataset:** IBM Telco Customer Churn — 7,043 customers, 21 features

---
## Table of Contents
1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Data Preprocessing](#3)
4. [Modeling — Baseline to XGBoost](#4)
5. [Model Evaluation](#5)
6. [SHAP Explainability](#6)
7. [Business Recommendation](#7)

## 1. Setup & Data Loading <a id='1'></a>

### Why these libraries?
- **pandas / numpy** — data manipulation
- **matplotlib / seaborn** — visualizations
- **scikit-learn** — ML models and evaluation tools
- **xgboost** — gradient boosting, one of the most requested skills in Istanbul DS jobs
- **shap** — explains *why* the model made each prediction (crucial for business use)
- **imbalanced-learn** — handles class imbalance (more on this later)

In [ ]:
!pip install xgboost shap imbalanced-learn -q
print("✅ Libraries installed!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score,
                              roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

# Plot style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("✅ All libraries imported!")

In [ ]:
# Load dataset
# If using Google Colab, upload the CSV file first or connect to Kaggle
telecom_cust = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(f"Shape: {telecom_cust.shape}")
print(f"Columns: {list(telecom_cust.columns)}")
telecom_cust.head()

## 2. Exploratory Data Analysis (EDA) <a id='2'></a>

**What is EDA?**
Before building any model, we need to *understand* the data. EDA helps us answer:
- What does the data look like?
- Are there missing values?
- Is the target variable balanced?
- Which features seem related to churn?

Good EDA directly informs better models.

In [ ]:
# Basic info
print("=== Dataset Info ===")
print(f"Rows: {telecom_cust.shape[0]}")
print(f"Columns: {telecom_cust.shape[1]}")
print(f"\nMissing values per column:")
print(telecom_cust.isnull().sum()[telecom_cust.isnull().sum() > 0])
print(f"\nData types:")
print(telecom_cust.dtypes.value_counts())

### 2.1 Churn Distribution

**Why this matters:** If 95% of customers don't churn, a model that always predicts "No churn" gets 95% accuracy — but it's completely useless. We call this the **class imbalance problem**. We'll handle it later with SMOTE.

In [ ]:
churn_counts = telecom_cust['Churn'].value_counts()
total = len(telecom_cust)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Not Churned (0)', 'Churned (1)'],
            churn_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, f'{v}\n({v/total*100:.1f}%)',
                 ha='center', fontweight='bold')
axes[0].set_title('Churn Distribution', fontweight='bold', pad=15)
axes[0].set_ylabel('Number of Customers')
axes[0].set_ylim(0, max(churn_counts.values) * 1.15)

# Pie chart
axes[1].pie(churn_counts.values, labels=['Not Churned', 'Churned'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Split', fontweight='bold', pad=15)

plt.suptitle('⚠️  Class Imbalance: ~73% vs ~27%', fontsize=13, y=1.02, color='#e74c3c')
plt.tight_layout()
plt.show()

print(f"Churn rate: {churn_counts[1]/total*100:.1f}%")
print(f"\n💡 Imbalanced dataset — we will apply SMOTE in preprocessing to fix this.")

### 2.2 Tenure vs Churn

**Tenure** = how many months the customer has been with the company.
We expect new customers to churn more — let's verify.

In [ ]:
bins = [0, 12, 24, 36, 48, 60, np.inf]
labels = ['0-12', '12-24', '24-36', '36-48', '48-60', '60+']
telecom_cust['tenure_group'] = pd.cut(telecom_cust['tenure'], bins=bins, labels=labels)

churn_rate = telecom_cust.groupby('tenure_group', observed=False)['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(churn_rate.index.astype(str), churn_rate.values,
              color=sns.color_palette("RdYlGn_r", len(churn_rate)), edgecolor='white')
for bar, v in zip(bars, churn_rate.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', fontweight='bold')
ax.set_title('Churn Rate by Tenure Group', fontweight='bold', pad=12)
ax.set_xlabel('Tenure (Months)')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, max(churn_rate.values) * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

print("💡 Customers in their first year churn the most — early intervention is critical!")

### 2.3 Contract Type vs Churn

**Contract type** is one of the most powerful predictors.
Month-to-month contracts give customers the freedom to leave anytime.

In [ ]:
churn_contract = telecom_cust.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax.bar(churn_contract.index, churn_contract.values,
              color=colors, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, churn_contract.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)
ax.set_title('Churn Rate by Contract Type', fontweight='bold', pad=12)
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, max(churn_contract.values) * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

print("💡 Month-to-month customers churn at 4x the rate of two-year contract customers!")

### 2.4 Monthly Charges vs Churn

Do higher-paying customers leave more? Let's check.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
not_churn_mc = telecom_cust[telecom_cust['Churn']=='No']['MonthlyCharges']
churn_mc     = telecom_cust[telecom_cust['Churn']=='Yes']['MonthlyCharges']
axes[0].boxplot([not_churn_mc, churn_mc], labels=['Not Churned', 'Churned'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Monthly Charges vs Churn', fontweight='bold')
axes[0].set_ylabel('Monthly Charges ($)')

# Distribution
axes[1].hist(not_churn_mc, bins=30, alpha=0.6, color='#2ecc71', label='Not Churned', density=True)
axes[1].hist(churn_mc, bins=30, alpha=0.6, color='#e74c3c', label='Churned', density=True)
axes[1].set_title('Monthly Charges Distribution', fontweight='bold')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"💡 Churned customers pay on average ${churn_mc.mean():.2f}/mo vs ${not_churn_mc.mean():.2f}/mo for retained customers.")

### 2.5 Correlation Heatmap

In [ ]:
# We need numeric df for correlation — build it first
df_corr = telecom_cust.copy()
df_corr['TotalCharges'] = pd.to_numeric(df_corr['TotalCharges'], errors='coerce').fillna(0)
df_corr['Churn_num'] = (df_corr['Churn'] == 'Yes').astype(int)
num_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_num']

corr = df_corr[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation Matrix (Numeric Features)', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print("💡 Tenure has a negative correlation with churn — longer customers are more loyal.")
print("💡 MonthlyCharges has a positive correlation — higher bills increase churn risk.")

## 3. Data Preprocessing <a id='3'></a>

**Steps:**
1. Fix data types
2. Encode categorical variables
3. Train/test split
4. Handle class imbalance with SMOTE
5. Scale features

**What is SMOTE?**
SMOTE (Synthetic Minority Over-sampling Technique) creates *synthetic* examples of the minority class (churners) so the model sees a balanced dataset during training. Without this, the model tends to just predict "No churn" for everyone.

In [ ]:
df = telecom_cust.copy()

# ── Fix TotalCharges type ──────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# ── Drop customerID (no predictive value) ─────────────────────
df = df.drop('customerID', axis=1)

# ── Encode target ──────────────────────────────────────────────
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── Drop helper columns created in EDA ───────────────────────
df = df.drop(columns=['tenure_group'], errors='ignore')

# ── One-hot encode all categorical columns ────────────────────
cat_cols = df.select_dtypes('object').columns.tolist()
print(f"Encoding {len(cat_cols)} categorical columns: {cat_cols}")
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# ── Convert bool to int ────────────────────────────────────────
bool_cols = df.select_dtypes('bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f"\n✅ Final shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
df.head(3)

In [ ]:
# ── Train / Test Split ────────────────────────────────────────
# Why stratify=True?
# Ensures both train and test sets have the same churn ratio (~27%)
# Without this, by chance one set could have very few churners

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size:  {X_test.shape[0]} rows")
print(f"\nTrain churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate:  {y_test.mean()*100:.1f}%  (stratified ✅)")

In [ ]:
# ── Apply SMOTE only on training data ─────────────────────────
# IMPORTANT: Never apply SMOTE to test data!
# Test data must reflect real-world distribution

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE — Train set: {y_train.value_counts().to_dict()}")
print(f"After SMOTE  — Train set: {pd.Series(y_train_sm).value_counts().to_dict()}")
print(f"\nNew train size: {X_train_sm.shape[0]} rows (balanced ✅)")

In [ ]:
# ── Scale features ────────────────────────────────────────────
# Why scale? Logistic Regression is sensitive to feature magnitude.
# XGBoost and Random Forest don't strictly need scaling, but it doesn't hurt.
# We fit the scaler on train data only, then transform both.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled  = scaler.transform(X_test)  # same scaler, don't refit!

print("✅ Features scaled using StandardScaler")
print(f"   Mean ~0: {X_train_scaled.mean():.4f}")
print(f"   Std  ~1: {X_train_scaled.std():.4f}")

## 4. Modeling <a id='4'></a>

**Our strategy:** Start simple, then increase complexity.

| Model | Why we use it |
|-------|--------------|
| **Logistic Regression** | Baseline — simple, interpretable, fast |
| **Random Forest** | Ensemble of decision trees, handles non-linearity |
| **XGBoost** | State-of-the-art gradient boosting — #1 requested ML tool in Istanbul |

A good data scientist always starts with a baseline before reaching for complex models. If Logistic Regression already gets 85% AUC, XGBoost adding 2% might not be worth the complexity.

### 4.1 Logistic Regression (Baseline)

**What is Logistic Regression?**
Despite the name, it's a *classification* algorithm. It models the probability of an event (churn) as a function of input features. It's linear, interpretable, and a standard baseline for binary classification.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_sm)

# Predictions
lr_pred  = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

lr_auc = roc_auc_score(y_test, lr_proba)
lr_acc = accuracy_score(y_test, lr_pred)

print(f"Logistic Regression Results")
print(f"{'─'*35}")
print(f"Accuracy : {lr_acc*100:.2f}%")
print(f"ROC-AUC  : {lr_auc:.4f}")
print(f"\n{classification_report(y_test, lr_pred, target_names=['No Churn', 'Churn'])}")

**How to read these metrics:**
- **Accuracy** — % of correct predictions. Misleading with imbalanced data!
- **Precision** — of all customers we flagged as churners, how many actually churned?
- **Recall** — of all actual churners, how many did we catch? *(More important for business!)*
- **F1-score** — balance between precision and recall
- **ROC-AUC** — overall model quality. 0.5 = random, 1.0 = perfect. Above 0.80 is good.

### 4.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)  # RF doesn't need scaled data

rf_pred  = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_auc = roc_auc_score(y_test, rf_proba)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"Random Forest Results")
print(f"{'─'*35}")
print(f"Accuracy : {rf_acc*100:.2f}%")
print(f"ROC-AUC  : {rf_auc:.4f}")
print(f"\n{classification_report(y_test, rf_pred, target_names=['No Churn', 'Churn'])}")

### 4.3 XGBoost

**What is XGBoost?**
XGBoost (Extreme Gradient Boosting) builds trees *sequentially* — each new tree focuses on correcting the mistakes of the previous ones. It's consistently the top performer on tabular data and is the most requested ML framework in Istanbul job postings.

**scale_pos_weight** handles class imbalance by telling XGBoost to pay more attention to the minority class (churners).

In [ ]:
# Calculate class weight ratio for XGBoost
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"Class weight ratio (scale_pos_weight): {scale:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,   # handles class imbalance
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

xgb_pred  = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

xgb_auc = roc_auc_score(y_test, xgb_proba)
xgb_acc = accuracy_score(y_test, xgb_pred)

print(f"\nXGBoost Results")
print(f"{'─'*35}")
print(f"Accuracy : {xgb_acc*100:.2f}%")
print(f"ROC-AUC  : {xgb_auc:.4f}")
print(f"\n{classification_report(y_test, xgb_pred, target_names=['No Churn', 'Churn'])}")

## 5. Model Evaluation <a id='5'></a>

Now let's compare all three models side by side with proper visualizations.

In [ ]:
# ── Model Comparison Table ────────────────────────────────────
from sklearn.metrics import f1_score

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [lr_acc, rf_acc, xgb_acc],
    'ROC-AUC':  [lr_auc, rf_auc, xgb_auc],
    'F1 (Churn)': [
        f1_score(y_test, lr_pred),
        f1_score(y_test, rf_pred),
        f1_score(y_test, xgb_pred)
    ]
}).set_index('Model').round(4)

print("📊 Model Comparison")
print("="*50)
print(results.to_string())
print(f"\n🏆 Best ROC-AUC: {results['ROC-AUC'].idxmax()} ({results['ROC-AUC'].max():.4f})")

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
models_roc = [
    ('Logistic Regression', lr_proba, '#3498db'),
    ('Random Forest',       rf_proba, '#2ecc71'),
    ('XGBoost',             xgb_proba,'#e74c3c'),
]
for name, proba, color in models_roc:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

axes[0].plot([0,1],[0,1], 'k--', alpha=0.4, label='Random (AUC=0.5)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — All Models', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Bar chart comparison
metrics = ['Accuracy', 'ROC-AUC', 'F1 (Churn)']
x = np.arange(len(metrics))
width = 0.25
colors = ['#3498db', '#2ecc71', '#e74c3c']
for i, (model, color) in enumerate(zip(results.index, colors)):
    axes[1].bar(x + i*width, results.loc[model], width, label=model,
                color=color, edgecolor='white')

axes[1].set_xticks(x + width)
axes[1].set_xticklabels(metrics)
axes[1].set_ylabel('Score')
axes[1].set_title('Model Comparison', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0.6, 1.0)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrix for Best Model (XGBoost) ─────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, xgb_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Not Churned', 'Churned'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('XGBoost — Confusion Matrix', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives  (correctly predicted stayed):  {tn}")
print(f"False Positives (predicted churn, actually stayed): {fp}")
print(f"False Negatives (missed churners — costly!):   {fn}")
print(f"True Positives  (correctly caught churners):   {tp}")
print(f"\n💡 For business: False Negatives are the most costly — churners we missed!")

In [ ]:
# ── Cross-Validation for robustness ───────────────────────────
# Cross-validation trains and evaluates the model on 5 different
# splits of the data to make sure the results aren't just lucky.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_cv = cross_val_score(xgb_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"XGBoost 5-Fold Cross-Validation ROC-AUC")
print(f"{'─'*40}")
for i, score in enumerate(xgb_cv):
    print(f"  Fold {i+1}: {score:.4f}")
print(f"{'─'*40}")
print(f"  Mean:  {xgb_cv.mean():.4f}")
print(f"  Std:   {xgb_cv.std():.4f}")
print(f"\n✅ Consistent results across folds — model is robust, not just lucky!")

## 6. SHAP Explainability <a id='6'></a>

**Why SHAP?**
A model that just says "this customer will churn" is not very useful. A business needs to know *why* — so they can act on it.

**SHAP (SHapley Additive exPlanations)** assigns each feature a contribution score for every individual prediction:
- **Positive SHAP value** → pushes prediction towards churn
- **Negative SHAP value** → pushes prediction away from churn

This turns a black-box model into an explainable tool that business teams can trust and act on.

In [ ]:
# Compute SHAP values for XGBoost
print("Computing SHAP values... (may take ~30 seconds)")
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
print("✅ SHAP values computed!")

### 6.1 Global Feature Importance — What Drives Churn Overall?

In [ ]:
# SHAP Summary Plot (Beeswarm)
# Each dot = one customer
# Color = feature value (red=high, blue=low)
# Position on x-axis = impact on churn prediction

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, plot_type='dot',
                  max_display=15, show=False)
plt.title('SHAP Beeswarm — Top 15 Features Driving Churn', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

**How to read this plot:**
- Features are ranked by importance (top = most impactful)
- Each dot represents one customer
- **Red dots on the right** = high feature value → increases churn probability
- **Blue dots on the left** = low feature value → decreases churn probability

In [ ]:
# SHAP Bar Plot — mean absolute impact
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type='bar',
                  max_display=15, show=False)
plt.title('Mean |SHAP| Value — Overall Feature Importance', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### 6.2 Individual Customer Explanation — Waterfall Plot

**Waterfall plots** show how each feature pushed a single prediction up or down from the baseline.
This is what you'd show a business stakeholder to explain a specific customer's churn risk.

In [ ]:
# Pick a high-risk customer (predicted to churn with high confidence)
high_risk_idx = np.argsort(xgb_proba)[::-1][0]
print(f"Customer index: {high_risk_idx}")
print(f"Predicted churn probability: {xgb_proba[high_risk_idx]*100:.1f}%")
print(f"Actual outcome: {'Churned ✅' if y_test.iloc[high_risk_idx]==1 else 'Did not churn ❌'}")

# Waterfall plot
shap_exp = shap.Explanation(
    values=shap_values[high_risk_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_risk_idx],
    feature_names=X_test.columns.tolist()
)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_exp, max_display=12, show=False)
plt.title(f'Why is this customer at risk? (Churn prob: {xgb_proba[high_risk_idx]*100:.1f}%)',
          fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 Dependence Plot — How Tenure Affects Churn

In [ ]:
# SHAP dependence plot shows how one feature's value relates to its SHAP impact
# Color shows interaction with another feature

shap.dependence_plot('tenure', shap_values, X_test,
                     interaction_index='MonthlyCharges', show=False)
plt.title('Tenure vs SHAP Value (colored by Monthly Charges)', fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Short tenure = high churn risk")
print("💡 High monthly charges + short tenure = extreme churn risk (red dots at left)")

## 7. Business Recommendation <a id='7'></a>

This is the most important section from a business perspective. A model is only valuable if it drives action.

Here we translate our findings into a concrete retention strategy with estimated financial impact.

In [ ]:
# ── At-Risk Customer List ─────────────────────────────────────
# Flag the top customers most likely to churn

risk_df = X_test.copy()
risk_df['churn_probability'] = xgb_proba
risk_df['actual_churn']      = y_test.values
risk_df['risk_tier'] = pd.cut(
    risk_df['churn_probability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print("📋 Risk Tier Distribution:")
print(risk_df['risk_tier'].value_counts())
print(f"\n🚨 High-risk customers: {(risk_df['risk_tier']=='High Risk').sum()}")

In [ ]:
# ── Financial Impact Estimate ─────────────────────────────────
# Assumptions (adjust based on real business context):
avg_monthly_revenue  = 65    # $ average monthly charge
avg_months_lost      = 24    # average customer lifetime lost when they churn
intervention_cost    = 25    # $ cost to offer a discount / retention call
intervention_success = 0.30  # 30% of contacted customers stay

high_risk = risk_df[risk_df['risk_tier'] == 'High Risk']
n_high_risk = len(high_risk)

revenue_at_risk = n_high_risk * avg_monthly_revenue * avg_months_lost
customers_saved = int(n_high_risk * intervention_success)
revenue_saved   = customers_saved * avg_monthly_revenue * avg_months_lost
total_cost      = n_high_risk * intervention_cost
net_benefit     = revenue_saved - total_cost

print("💰 FINANCIAL IMPACT ESTIMATE")
print("="*45)
print(f"High-risk customers identified:    {n_high_risk}")
print(f"Revenue at risk (if no action):   ${revenue_at_risk:,.0f}")
print(f"Customers saved (est. 30%):        {customers_saved}")
print(f"Revenue saved:                    ${revenue_saved:,.0f}")
print(f"Cost of intervention:             ${total_cost:,.0f}")
print(f"Net benefit:                      ${net_benefit:,.0f}")
print("="*45)
print(f"\n📈 ROI: {net_benefit/total_cost*100:.0f}%")

In [ ]:
# ── Top Retention Recommendations ─────────────────────────────
print("RETENTION STRATEGY")
print("1. TARGET: Month-to-month customers in first 12 months")
print("   Highest churn rate ~47%. Offer annual contract upgrade incentive.")
print("2. TARGET: High monthly charges + short tenure")
print("   SHAP shows this is strongest churn signal. Offer loyalty discount before month 6.")
print("3. TARGET: Fiber optic customers — higher churn than DSL.")
print("   Investigate service quality, offer credits.")
print("4. TARGET: Customers without online security/tech support.")
print("   Bundle at discount for at-risk customers.")
print("5. AUTOMATE: Deploy model to score customers monthly.")
print("   Trigger outreach for anyone above 60% churn probability.")

In [ ]:
# ── Final Model Summary ───────────────────────────────────────
print("📊 FINAL MODEL SUMMARY")
print("="*45)
print(f"Best Model:      XGBoost")
print(f"ROC-AUC:         {xgb_auc:.4f}")
print(f"CV ROC-AUC:      {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")
print(f"Churn Recall:    {__import__('sklearn.metrics',fromlist=['recall_score']).recall_score(y_test, xgb_pred):.4f}")
print(f"\nTop 3 Churn Drivers (from SHAP):")
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test.columns)
top3 = mean_shap.nlargest(3)
for i, (feat, val) in enumerate(top3.items(), 1):
    print(f"  {i}. {feat} (mean |SHAP| = {val:.4f})")
print(f"\n✅ Notebook complete — ready for portfolio!")